In [ ]:
import scipy.signal as signal
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import textwrap
import os
import sys
import numpy as np
import pandas as pd
import importlib
import h5py

In [ ]:
def print_structure(name, obj):
    print(f"{name} is a {type(obj)}")

with h5py.File('wille_simulated_data.h5', 'r') as f:
    # Assuming the dataset is a 2D array: (number_of_spectra, channels)
    data = f['data/block0_values'][:]

# Define combined statistics from the raw captured spectra source
npz_path = 'bighorntest3.npz'  # change to desired NPZ
with np.load(npz_path) as npz:
    if 'data' in npz.files:
        arr = np.asarray(npz['data'])
    else:
        arr = np.asarray(npz[npz.files[0]])

# Squeeze and normalize dims to 2D (measurements, channels)
arr = np.squeeze(arr)
if arr.ndim == 1:
    arr = arr[np.newaxis, :]
elif arr.ndim == 3:
    # assume last axis is channels; otherwise try to place channel axis last
    if arr.shape[-1] < arr.shape[0]:
        arr = arr.reshape(-1, arr.shape[-1])
    else:
        arr = arr.reshape(-1, arr.shape[-1])
elif arr.ndim != 2:
    raise ValueError(f'NPZ array has unexpected ndim={arr.ndim}')

# Convert to numeric if necessary
if not np.issubdtype(arr.dtype, np.number):
    arr = arr.astype(float)

# Compute combined statistics across measurements (axis=0)
combined_mean_npz = np.mean(arr, axis=0)
combined_median_npz = np.median(arr, axis=0)

n_channels_npz = combined_mean_npz.size
frequencies_npz = np.linspace(1419.5, 1421.5, n_channels_npz)

plt.figure(figsize=(10, 6))
plt.plot(frequencies_npz, combined_mean_npz, label='npz mean (Less noisy)', linestyle='--', color='tab:blue')
plt.plot(frequencies_npz, combined_median_npz, label='npz median (Robust)', alpha=0.7, color='tab:red')
plt.title('Combined Spectral Averaging (NPZ): Mean vs Median')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Power')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()

# Optionally compare NPZ combined to HDF5 combined if available

# Simple NPZ combined mean/median overlay
npz_path = 'bighorntest3.npz'  # change to desired NPZ file
with np.load(npz_path) as z:
    if 'data' in z.files:
        arr = np.asarray(z['data'])
    else:
        arr = np.asarray(z[z.files[0]])

# Normalize shape to (measurements, channels)
arr = np.squeeze(arr)
if arr.ndim == 3:
    arr = arr.reshape(-1, arr.shape[-1])
if arr.ndim == 1:
    arr = arr[np.newaxis, :]

# Compute mean and median across measurements
mean_spec = np.mean(arr, axis=0)
median_spec = np.median(arr, axis=0)

# Frequency axis (adjust if your band differs)
freqs = np.linspace(1419.5, 1421.5, mean_spec.size)

plt.figure(figsize=(10,6))
plt.plot(freqs, mean_spec, label='mean', linestyle='--', color='C0')
plt.plot(freqs, median_spec, label='median', alpha=0.8, color='C1')
plt.title('NPZ Combined Mean vs Median')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Power')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()
if 'combined_mean' in globals():
    plt.figure(figsize=(10,6))
    # resample or align lengths if they differ
    m1 = combined_mean if 'combined_mean' in globals() else None
    m2 = combined_mean_npz
    if m1 is not None and m1.size != m2.size:
        # simple interpolation of m2 to m1 freq grid
        f_h5 = np.linspace(1419.5, 1421.5, m1.size)
        f_npz = np.linspace(1419.5, 1421.5, m2.size)
        m2_interp = np.interp(f_h5, f_npz, m2)
        plt.plot(f_h5, m1, label='HDF5 combined_mean', color='tab:green')
        plt.plot(f_h5, m2_interp, label='NPZ combined_mean (interp)', color='tab:purple')
    else:
        plt.plot(np.linspace(1419.5, 1421.5, m2.size), m2, label='NPZ combined_mean', color='tab:purple')
        if m1 is not None:
            plt.plot(np.linspace(1419.5, 1421.5, m1.size), m1, label='HDF5 combined_mean', color='tab:green')
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Power')
    plt.title('Compare combined_mean: HDF5 vs NPZ')
    plt.legend()
    plt.grid(True, linestyle=':')
    plt.show()

In [ ]:
# Load data from an .npz file and plot a waterfall (measurements x channels)
npz_path = 'bighorntest3.npz'  # change this to the NPZ file you want to use
with np.load(npz_path) as npz:
    # Prefer an entry named 'data' if present, otherwise take the first array
    if 'data' in npz.files:
        npz_data = npz['data']
    else:
        keys = [k for k in npz.files]
        npz_data = npz[keys[0]]

# Convert and squeeze singleton dims
npz_data = np.asarray(npz_data)
npz_data = np.squeeze(npz_data)

# Normalize to 2D: (measurements, channels)
if npz_data.ndim == 1:
    npz_data = npz_data[np.newaxis, :]
elif npz_data.ndim == 3:
    # Try to detect which axis is channels. Prefer matching the HDF5 'data' channels if available
    ref_channels = None
    if 'data' in globals() and isinstance(data, np.ndarray):
        ref_channels = data.shape[1] if data.ndim >= 2 else None
    # If any axis matches ref_channels, move it to last and merge the other two axes into measurements
    if ref_channels is not None and ref_channels in npz_data.shape:
        ch_axis = int(np.where(np.array(npz_data.shape) == ref_channels)[0][0])
        # move channels to last axis
        if ch_axis != 2:
            npz_data = np.moveaxis(npz_data, ch_axis, 2)
        npz_data = npz_data.reshape(-1, npz_data.shape[2])
    else:
        # assume last axis is channels (common: blocks x measurements x channels)
        npz_data = npz_data.reshape(-1, npz_data.shape[2])
elif npz_data.ndim != 2:
    raise ValueError(f'NPZ contains array with {npz_data.ndim} dims; expected 1, 2 or 3')

# If an HDF5 `data` variable exists, try to match its orientation
if 'data' in globals() and isinstance(data, np.ndarray):
    if npz_data.shape == data.T.shape and npz_data.shape != data.shape:
        npz_data = npz_data.T

# Heuristic: if rows > cols it's likely transposed (channels x measurements)
rows, cols = npz_data.shape
if rows > cols:
    npz_data = npz_data.T
    rows, cols = npz_data.shape

# Ensure numeric dtype
if not np.issubdtype(npz_data.dtype, np.number):
    try:
        npz_data = npz_data.astype(float)
    except Exception as e:
        raise TypeError(f'Could not convert NPZ data to numeric dtype: {e}')

print(f'Loaded {npz_path} with shape {npz_data.shape}')

frequencies_npz = np.linspace(1419.5, 1421.5, npz_data.shape[1])
plt.figure(figsize=(10, 8))
im = plt.imshow(npz_data,
                aspect='auto',
                extent=[frequencies_npz[0], frequencies_npz[-1], npz_data.shape[0], 0],
                cmap='viridis',
                interpolation='none')
cbar = plt.colorbar(im)
cbar.set_label('Power')
plt.title(f'Waterfall Plot (NPZ): {npz_path}')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Measurement Number (Time)')
plt.show()
midpoint = npz_data.shape[0] // 2
s_on_data = npz_data[:midpoint, :]
s_off_data = npz_data[midpoint:, :]

# --- 2. AVERAGE (To reduce the "grainy" noise) ---
# Your lab manual notes: "the resulting spectrum will look noisy... 
# combining many gives a less noisy result."
s_on_avg = np.mean(s_on_data, axis=0)
s_off_avg = np.mean(s_off_data, axis=0)

# --- 3. REVEAL THE SIGNAL (Difference and Ratio) ---
# Difference: Removes the "fixed pattern" vertical stripes
difference = s_on_avg - s_off_avg

# Ratio: Often cleaner as it accounts for gain variations
# We subtract 1 to center the baseline at 0
ratio = (s_on_avg / s_off_avg) - 1

# --- 4. PLOTTING ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Plot Difference Spectrum
ax1.plot(frequencies_npz, difference, color='tab:red', label='ON - OFF (Difference)')
ax1.set_ylabel('Power Difference')
ax1.set_title('Revealing the HI Line: Difference vs. Ratio')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot Ratio Spectrum
ax2.plot(frequencies_npz, ratio, color='tab:blue', label='(ON / OFF) - 1')
ax2.set_xlabel('Frequency (MHz)')
ax2.set_ylabel('Relative Intensity')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

from scipy.ndimage import gaussian_filter
baseline = np.median(npz_data, axis=0)

# 2. Subtract the baseline from every row
# This 'flattens' the instrumental hump and removes static vertical noise
diff_waterfall = npz_data - baseline
# 1. Apply a Gaussian blur to the differential data
# sigma=(time_blur, frequency_blur)
# We blur more in time (vertical) to find the consistent signal
smoothed_diff = gaussian_filter(diff_waterfall, sigma=(50, 2))

plt.figure(figsize=(12, 8))

# 2. Re-plot with the smoothed data
im = plt.imshow(smoothed_diff,
                aspect='auto',
                extent=[frequencies_npz[0], frequencies_npz[-1], npz_data.shape[0], 0],
                cmap='RdBu_r', 
                interpolation='bilinear')

# 3. Aggressively tighten the color limits
# This 'squashes' the noise and forces faint signals to show color
v_limit = np.std(smoothed_diff) * 1.5 
im.set_clim(-v_limit, v_limit)

plt.colorbar(im, label='Smoothed Relative Power')
plt.title('Revealing the HI Smear (Gaussian Smoothed)')
plt.show()



# 3. Apply a small amount of Smoothing (optional but helps with the 'smear')
# If the data is too grainy, the line stays hidden. 
# Here we use a basic Gaussian filter if you have scipy, or just show raw.

plt.figure(figsize=(12, 8))

# Use a 'Diverging' colormap (like RdBu_r or coolwarm) 
# This makes features 'brighter' than the average look red/yellow 
# and 'darker' look blue.
norm = colors.TwoSlopeNorm(vcenter=0) # Centers the color scale at 0

im = plt.imshow(diff_waterfall,
                aspect='auto',
                extent=[frequencies_npz[0], frequencies_npz[-1], npz_data.shape[0], 0],
                cmap='RdBu_r', 
                norm=norm,
                interpolation='bilinear') # 'bilinear' helps create that 'smear' look

cbar = plt.colorbar(im)
cbar.set_label('Power relative to Median')
plt.xlim(1420.1, 1420.8)

# Even heavier vertical smoothing to kill the noise
smoothed_diff_heavy = gaussian_filter(diff_waterfall, sigma=(100, 1))

# Re-run your imshow with these tighter limits
im.set_data(smoothed_diff_heavy)

# 2. Re-calculate v_limit based on the SMOOTHED data
# Using 2 or 3 standard deviations of the smoothed data helps features "pop"
v_limit_smoothed = np.std(smoothed_diff_heavy) * 2

# 3. Apply the new limits (This replaces the previous im.set_clim and v_limit lines)
im.set_clim(-v_limit_smoothed, v_limit_smoothed)

# 4. Ensure the x-axis zoom is applied correctly
plt.xlim(1420.1, 1420.8)

plt.show()

In [ ]:
# Compute temporal power spectra (Welch) for NPZ readings
# Uses the existing `npz_data` if present, otherwise loads the NPZ file
npz_path = 'bighorntest3.npz'
try:
    _npz = npz_data
except NameError:
    with np.load(npz_path) as npz:
        if 'data' in npz.files:
            _npz = npz['data']
        else:
            _npz = npz[npz.files[0]]

# Normalize and reshape like the waterfall cell does
_npz = np.asarray(_npz)
_npz = np.squeeze(_npz)
if _npz.ndim == 3:
    _npz = _npz.reshape(-1, _npz.shape[-1])
if _npz.ndim == 1:
    _npz = _npz[np.newaxis, :]
if _npz.ndim != 2:
    raise ValueError(f'Unexpected array shape for PSD computation: {_npz.shape}')

# Choose channel (center) and compute Welch PSD (temporal) for that channel
ch_idx = _npz.shape[1] // 2
timeseries = _npz[:, ch_idx]
from scipy.signal import welch
fs = 1.0  # sampling rate (samples per measurement); set if you know the real rate
nperseg = min(256, max(8, len(timeseries)))
f, Pxx = welch(timeseries, fs=fs, nperseg=nperseg)
plt.figure(figsize=(8, 4))
plt.semilogy(f, Pxx, label=f'Channel {ch_idx}')
plt.title('Welch PSD — single channel')
plt.xlabel('Frequency (cycles per sample)')
plt.ylabel('PSD')
plt.grid(True, which='both', ls=':')
plt.legend()

# Average PSD across a small band around the center channel
halfband = 5
start = max(0, ch_idx - halfband)
stop = min(_npz.shape[1], ch_idx + halfband + 1)
psd_list = []
for c in range(start, stop):
    _, p = welch(_npz[:, c], fs=fs, nperseg=nperseg)
    psd_list.append(p)
psd_mean = np.mean(psd_list, axis=0)
plt.figure(figsize=(8, 4))
plt.semilogy(f, psd_mean, color='tab:orange', label=f'Avg channels {start}-{stop-1}')
plt.title('Welch PSD — averaged band')
plt.xlabel('Frequency (cycles per sample)')
plt.ylabel('PSD')
plt.grid(True, which='both', ls=':')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
on_target_spectrum = np.mean(smoothed_diff_heavy[:midpoint, :], axis=0)

# 2. Create the plot
plt.figure(figsize=(10, 6))

# Plot the 1D Power Spectrum
plt.plot(frequencies_npz, on_target_spectrum, color='red', linewidth=2, label='Smoothed HI Signal')

# Formatting for your report
plt.axhline(0, color='black', linewidth=1, alpha=0.5) # Baseline
plt.axvline(1420.406, color='blue', linestyle='--', alpha=0.7, label='HI Rest Freq')

# Zoom in on the area where you saw the red/blue smear
plt.xlim(1420.2, 1420.7) 

plt.title('Power Spectrum of Isolated HI Signal')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Relative Power (Smoothed)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
from scipy.ndimage import uniform_filter1d

# 1. Define midpoint if not already defined
midpoint = npz_data.shape[0] // 2 

# 2. Get the ON-target spectrum from your differential data
# This is the 1D line that contains your "red" smear
on_target_raw = np.mean(diff_waterfall[:midpoint, :], axis=0)

# 3. Create 'smoothed_peak'
# A size of 10-15 is a good "compromise" for the 1 km/s rule
smoothed_peak = uniform_filter1d(on_target_raw, size=15)

# 4. Constants for Doppler conversion
f0 = 1420.406  # Rest frequency in MHz
c = 299792.458 # km/s

# 5. Convert frequencies to Velocity (km/s)
velocities = c * (f0 - frequencies_npz) / f0

# 6. Plotting
plt.figure(figsize=(10, 6))
plt.plot(velocities, smoothed_peak, color='darkred', linewidth=2, label='HI Profile')

plt.axvline(0, color='black', linestyle='--', alpha=0.5, label='Rest (0 km/s)')
plt.xlim(-150, 150) # Galaxy-standard zoom
plt.axhline(0, color='black', linewidth=0.5)

plt.title('HI Line Velocity Profile')
plt.xlabel('Velocity (km/s)')
plt.ylabel('Relative Power')
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
%matplotlib qt 
# data = np.load('bighorntest3.npz')  # Ensure this file exists and has the expected structure
# print("keys:", data.files)
def plot_waterfall(file_path, array_name='data'):
    # 1. Load the data
    with np.load(file_path) as data:
        # Assuming data is a 2D array: (time_steps, frequency_bins)
        matrix = data[array_name]
        
    matrix = np.squeeze(matrix) 

    if matrix.ndim != 2:
        print(f"Warning: Data has {matrix.ndim} dimensions: {matrix.shape}")
        # If it's 3D, just take the first 'slice' (e.g., the first channel)
        if matrix.ndim == 3:
            matrix = matrix[:, :, 0]

    num_rows, num_cols = matrix.shape
    
    # 2. Create X and Y axes
    x = np.arange(num_cols)  # e.g., Frequency
    y = np.arange(num_rows)  # e.g., Time
    
    verts = []
    for i in range(num_rows):
        # Create a polygon for each row that dips to zero at the edges
        # This prevents the "floating line" look and fills the area
        xs = np.concatenate([[x[0]], x, [x[-1]]])
        ys = np.concatenate([[0], matrix[i, :], [0]])
        verts.append(list(zip(xs, ys)))

    # 3. Setup the 3D plot
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    # Create the PolyCollection
    poly = PolyCollection(verts, facecolors=plt.cm.viridis(np.linspace(0, 1, num_rows)), alpha=0.7)
    
    # Add the polygons to the 3D plot along the Y axis
    ax.add_collection3d(poly, zs=y, zdir='y')

    # 4. Set plot limits (Crucial for PolyCollection)
    ax.set_xlim(0, num_cols)
    ax.set_ylim(0, num_rows)
    ax.set_zlim(np.min(matrix), np.max(matrix))

    ax.set_xlabel('Frequency Bins')
    ax.set_ylabel('Time / Frame')
    ax.set_zlabel('Amplitude')
    
    plt.title(f"Waterfall Plot: {file_path}")
    plt.show()


plot_waterfall('bighorntest3.npz', 'samples')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from scipy.ndimage import gaussian_filter

def plot_3d_hi_waterfall(npz_path, array_key='samples'):
    # 1. Load and Process Data
    with np.load(npz_path) as data:
        # Load the main matrix
        matrix = np.abs(data[array_key]) 

    matrix = np.squeeze(matrix) 
    if matrix.ndim == 3:
        matrix = matrix[:, :, 0]    
    # Baseline Subtraction
    baseline = np.median(matrix, axis=0)
    diff_waterfall = matrix - baseline
    
    # Gaussian Smoothing (Adjusted for 3D visibility)
    # Sigma (Time, Frequency)
    smoothed = gaussian_filter(diff_waterfall, sigma=(5, 1)) 
    
    # 2. Handle the Missing Frequency Key
    num_rows, num_cols = smoothed.shape
    
    # Manually define your frequency range here (e.g., 1420.1 to 1420.8 MHz)
    f_start, f_end = 1420.1, 1420.8
    x_axis = np.linspace(f_start, f_end, num_cols)
    
    # 3. Downsample for 3D Performance
    # Increase these if the plot is laggy
    row_step = 10 
    col_step = 1
    
    plot_data = smoothed[::row_step, ::col_step]
    x_plot = x_axis[::col_step]
    y_plot = np.arange(plot_data.shape[0])

    # 4. Prepare Polygons
    verts = []
    for i in range(plot_data.shape[0]):
        # "Close" the polygon at the baseline (0) for the filled look
        xs = np.concatenate([[x_plot[0]], x_plot, [x_plot[-1]]])
        ys = np.concatenate([[0], plot_data[i, :], [0]])
        verts.append(list(zip(xs, ys)))

    # 5. Create the 3D Plot
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Apply the RdBu_r colormap to the slices
    poly = PolyCollection(verts, 
                          facecolors=plt.cm.RdBu_r(np.linspace(0.3, 0.8, len(verts))), 
                          alpha=0.7, 
                          edgecolor='black', 
                          lw=0.3)
    
    ax.add_collection3d(poly, zs=y_plot, zdir='y')

    # 6. Set Limits and Orientation
    ax.set_xlim(x_plot.min(), x_plot.max())
    ax.set_ylim(0, len(y_plot))
    
    # Standard deviation based scaling for the Z-axis (height)
    v_limit = np.std(plot_data) * 4
    ax.set_zlim(-v_limit, v_limit)

    ax.set_xlabel('Frequency (Estimated MHz)')
    ax.set_ylabel('Time (Slices)')
    ax.set_zlabel('Relative Intensity')
    
    # Initial Rotation
    ax.view_init(elev=25, azim=-60)
    
    plt.title(f'3D HI Smear: {array_key}')
    plt.show()

# plot_3d_hi_waterfall('bighorntest3.npz', 'samples')
# Run it
plot_3d_hi_waterfall('bighorntest3.npz')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

def plot_2d_waterfall(file_path, array_key='samples'):
    # 1. Load Data
    with np.load(file_path) as data:
        matrix = np.abs(data[array_key])

    # 2. Fix Shape / Rank (Prevents "sequence argument" error)
    matrix = np.squeeze(matrix) # Remove empty dimensions
    
    if matrix.ndim == 1:
        # If it's just one row, we can't do a waterfall, so we expand it
        matrix = matrix[np.newaxis, :]
    elif matrix.ndim == 3:
        # If it has a channel dim (Time, Freq, 1), take the first slice
        matrix = matrix[:, :, 0]

    # 3. Baseline Subtraction (Removes the vertical lines/static)
    baseline = np.median(matrix, axis=0)
    diff_waterfall = matrix - baseline

    # 4. Apply Gaussian Smoothing
    # We use a sigma of (0.5, 1) to keep it looking crisp like your image
    smoothed = gaussian_filter(diff_waterfall, sigma=(0.5, 1))

    # 5. Plotting
    plt.figure(figsize=(10, 6))
    
    # Define frequency range based on your image (1419.5 to 1421.5)
    f_start, f_end = 1419.5, 1421.5
    
    im = plt.imshow(smoothed, 
                    aspect='auto', 
                    extent=[f_start, f_end, matrix.shape[0], 0],
                    cmap='viridis', # Matches your image
                    interpolation='bilinear')

    # 6. Color Scaling (The "Secret Sauce")
    # Setting the limits to +/- a few standard deviations makes faint signals pop
    v_limit = np.std(smoothed) * 2
    im.set_clim(-v_limit, v_limit)

    plt.colorbar(im, label='Relative Power')
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Measurement Number (Time)')
    plt.title(f'Waterfall Plot (NPZ): {file_path}')
    
    plt.show()

# Run the function
plot_2d_waterfall('bighorntest3.npz', 'samples')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_hi_spectrum_with_fft(file_path, fs=2.5e6, center_freq_mhz=1420.0):
    # 1. Load the data
    with np.load(file_path) as data:
        raw_data = data['samples'] 
    
    # 2. Reconstruct Complex Signal
    # Shape: (1000, 4096)
    complex_data = raw_data[..., 0] + 1j * raw_data[..., 1]

    # 3. Frequency Conversion (If the 4096 is NOT already an FFT)
    # If the second dimension is time-series, we run:
    power_spectrum = np.fft.fft(complex_data, axis=1)
    # If it IS already frequency bins, we just take the power:
   # power_spectrum = np.abs(complex_data)**2 

    # 4. Average over the Time axis (Integrate)
    # Shape: (4096,)
    integrated_spectrum = np.mean(power_spectrum, axis=0)

    # 5. Shift and Frequency Axis
    # fftshift is critical; it puts the center frequency in the center
    integrated_spectrum = np.fft.fftshift(integrated_spectrum)
    
    freq_offsets = np.linspace(-fs/2, fs/2, len(integrated_spectrum))
    sky_freqs = (freq_offsets / 1e6) + center_freq_mhz

    # 6. Plotting with a wider Y-axis perspective
    plt.figure(figsize=(10, 6))
    plt.plot(sky_freqs, 10 * np.log10(integrated_spectrum), color='navy', lw=0.8)
    
    plt.axvline(1420.406, color='red', linestyle='--', alpha=0.7, label='HI Rest (1420.4 MHz)')
    
    plt.title(f"Power Spectrum: {file_path}")
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Power (dB - relative)")
    plt.grid(True, alpha=0.2)
    plt.legend()
    
    # Force the Y-axis to show more than 0.07dB to see the 'shape'
    plt.show()

plot_hi_spectrum_with_fft("bighorn_wall.npz", fs=2e6, center_freq_mhz=1420.0)